<a href="https://colab.research.google.com/github/eyos52/Shahed-Drone-Detection-Targeting/blob/main/shahedRealWorldTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shahed-136 YOLO: Real-World Testing Notebook

Tests a trained YOLO model on real images. Based on `real_eval.py` (same defaults: `iou=0.45`, `conf=0.20`).

**Without labels** you get: detection rate, confidence distribution, threshold sweep, speed, annotated gallery.
**With YOLO-format labels** (optional) you also get: precision, recall, F1, AP50, and a list of false positives and misses.

Design note: the model runs **once** at a very low confidence floor. All thresholds are then applied by filtering. This is equivalent to rerunning at each threshold, because NMS only removes boxes that overlap a higher-confidence box.

## 1. Configuration

In [ ]:
from pathlib import Path

# ---- Paths (defaults match real_eval.py) ----
modelPath = "/users/apgoel/runs/shahed_ft_real/weights/best.pt"
sourceDir = "/users/apgoel/data/real_images"
outputDir = "/users/apgoel/runs/real_world_eval_nb"

# Optional ground truth: folder of YOLO .txt files named like the images.
# Set to None to skip the labeled evaluation.
labelsDir = None  # example: "/users/apgoel/data/real_labels"

# ---- Inference settings ----
confFloor = 0.001        # lowest confidence kept for analysis
confMain = 0.20          # main threshold (same as real_eval.py)
confSweep = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
iouNms = 0.45            # NMS IoU (same as real_eval.py)
iouMatch = 0.50          # IoU needed to count a detection as correct (labeled eval)
imgSize = 640
galleryCount = 12        # number of images shown in the gallery
imageExts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

outputPath = Path(outputDir)
outputPath.mkdir(parents=True, exist_ok=True)

## 2. Imports and model

In [ ]:
!pip install -q ultralytics

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
from ultralytics import YOLO

# AT&T brand colors
attBlue = "#009FDB"
attCobalt = "#0057B8"
attNavy = "#00388F"
attGray = "#6D6E71"
attOrange = "#F68D2E"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

assert Path(modelPath).exists(), f"Model not found: {modelPath}. Please upload your model and update modelPath in Section 1."
assert Path(sourceDir).exists(), f"Image folder not found: {sourceDir}. Please upload your images and update sourceDir in Section 1."

model = YOLO(modelPath)
print("Classes:", model.names)

imagePaths = sorted(p for p in Path(sourceDir).rglob("*") if p.suffix.lower() in imageExts)
print(f"Found {len(imagePaths)} images in {sourceDir}")
assert len(imagePaths) > 0, "No images found"

AssertionError: Model not found: /users/apgoel/runs/shahed_ft_real/weights/best.pt. Please upload your model and update modelPath in Section 1.

## 3. Run inference once at a low confidence floor

In [ ]:
startTime = time.time()
results = model.predict(
    source=[str(p) for p in imagePaths],
    conf=confFloor,
    iou=iouNms,
    imgsz=imgSize,
    save=False,
    verbose=False,
)
print(f"Inference done in {time.time() - startTime:.1f} s for {len(results)} images")

# Store all predictions in plain Python structures
preds = []
for imgPath, r in zip(imagePaths, results):
    h, w = r.orig_shape
    preds.append({
        "path": imgPath,
        "name": imgPath.name,
        "width": w,
        "height": h,
        "conf": r.boxes.conf.cpu().numpy(),
        "cls": r.boxes.cls.cpu().numpy().astype(int),
        "xyxy": r.boxes.xyxy.cpu().numpy(),
        "xyxyn": r.boxes.xyxyn.cpu().numpy(),
        "speed": r.speed,
    })

## 4. Detection summary at the main threshold

In [ ]:
rows = []
for p in preds:
    keep = p["conf"] >= confMain
    confs = p["conf"][keep]
    rows.append({
        "image": p["name"],
        "nDet": int(keep.sum()),
        "maxConf": float(confs.max()) if len(confs) else 0.0,
        "maxConfAnyThreshold": float(p["conf"].max()) if len(p["conf"]) else 0.0,
        "preprocessMs": p["speed"]["preprocess"],
        "inferenceMs": p["speed"]["inference"],
        "postprocessMs": p["speed"]["postprocess"],
    })
dfImages = pd.DataFrame(rows)

nImages = len(dfImages)
nDetected = int((dfImages["nDet"] > 0).sum())
print(f"Threshold {confMain}: {nDetected}/{nImages} images with at least 1 detection "
      f"({100 * nDetected / nImages:.1f} %)")
print(f"Total detections: {int(dfImages['nDet'].sum())}")
print(f"Images with more than 1 detection: {int((dfImages['nDet'] > 1).sum())}")

display(dfImages.sort_values("maxConfAnyThreshold").head(15))
dfImages.to_csv(outputPath / "perImageSummary.csv", index=False)

Images with `nDet == 0` but a nonzero `maxConfAnyThreshold` are near misses. Lowering the threshold would detect them, with more false positives.

## 5. Threshold sweep (detection rate)

In [ ]:
sweepRows = []
for t in confSweep:
    nDetImgs = sum(1 for p in preds if (p["conf"] >= t).any())
    nBoxes = sum(int((p["conf"] >= t).sum()) for p in preds)
    sweepRows.append({
        "conf": t,
        "imagesWithDet": nDetImgs,
        "detectionRate": nDetImgs / nImages,
        "totalBoxes": nBoxes,
        "boxesPerImage": nBoxes / nImages,
    })
dfSweep = pd.DataFrame(sweepRows)
display(dfSweep)
dfSweep.to_csv(outputPath / "thresholdSweep.csv", index=False)

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(dfSweep["conf"], dfSweep["detectionRate"], marker="o", color=attCobalt, label="Detection rate")
ax1.set_xlabel("Confidence threshold")
ax1.set_ylabel("Images with at least 1 detection", color=attCobalt)
ax1.set_ylim(0, 1.05)
ax2 = ax1.twinx()
ax2.plot(dfSweep["conf"], dfSweep["boxesPerImage"], marker="s", color=attOrange, label="Boxes per image")
ax2.set_ylabel("Boxes per image", color=attOrange)
ax2.grid(False)
ax2.spines["right"].set_visible(True)
ax1.set_title("Detection rate vs confidence threshold")
fig.tight_layout()
fig.savefig(outputPath / "thresholdSweep.png")
plt.show()

Read the curve like this: a stable detection rate across thresholds means the model is confident on real images. A sharp drop means the detections are weak. Boxes per image above about 1 at high thresholds can mean duplicate or false detections. Without labels this does not prove correctness.

## 6. Confidence distribution and speed

In [ ]:
allConf = np.concatenate([p["conf"] for p in preds]) if preds else np.array([])
topConf = dfImages["maxConfAnyThreshold"].values

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(allConf[allConf >= 0.05], bins=30, color=attBlue, edgecolor="white")
axes[0].axvline(confMain, color=attNavy, linestyle="--", label=f"conf = {confMain}")
axes[0].set_title("All box confidences (>= 0.05)")
axes[0].set_xlabel("Confidence")
axes[0].set_ylabel("Boxes")
axes[0].legend()

axes[1].hist(topConf, bins=20, range=(0, 1), color=attCobalt, edgecolor="white")
axes[1].axvline(confMain, color=attNavy, linestyle="--", label=f"conf = {confMain}")
axes[1].set_title("Top confidence per image")
axes[1].set_xlabel("Confidence")
axes[1].set_ylabel("Images")
axes[1].legend()
fig.tight_layout()
fig.savefig(outputPath / "confidenceDistribution.png")
plt.show()

speedDf = dfImages[["preprocessMs", "inferenceMs", "postprocessMs"]]
speedDf = speedDf.assign(totalMs=speedDf.sum(axis=1))
display(speedDf.describe().loc[["mean", "50%", "max"]].round(2))
print(f"Approx. throughput: {1000 / speedDf['totalMs'].mean():.1f} images/s (includes first-call warmup)")

## 7. Annotated gallery

In [ ]:
def loadGt(imgName, width, height):
    # Returns GT boxes as pixel xyxy, or None when labelsDir is not set
    if labelsDir is None:
        return None
    labelFile = Path(labelsDir) / (Path(imgName).stem + ".txt")
    if not labelFile.exists():
        return np.zeros((0, 4))
    boxes = []
    for line in labelFile.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        xc, yc, bw, bh = (float(v) for v in parts[1:5])
        boxes.append([(xc - bw / 2) * width, (yc - bh / 2) * height,
                      (xc + bw / 2) * width, (yc + bh / 2) * height])
    return np.array(boxes).reshape(-1, 4)


def drawImage(ax, p, thr):
    img = Image.open(p["path"]).convert("RGB")
    ax.imshow(img)
    keep = p["conf"] >= thr
    for box, c in zip(p["xyxy"][keep], p["conf"][keep]):
        x1, y1, x2, y2 = box
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor=attBlue, linewidth=2))
        ax.text(x1, max(y1 - 4, 0), f"{c:.2f}", color="white", fontsize=8,
                bbox=dict(facecolor=attCobalt, pad=1, edgecolor="none"))
    gt = loadGt(p["name"], p["width"], p["height"])
    if gt is not None:
        for x1, y1, x2, y2 in gt:
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                           edgecolor=attOrange, linewidth=2, linestyle="--"))
    ax.set_title(p["name"][:28], fontsize=8)
    ax.axis("off")


# Show a mix: lowest top-confidence images first (hardest cases), then the rest
order = np.argsort(topConf)
pick = list(order[:galleryCount])
nCols = 4
nRows = int(np.ceil(len(pick) / nCols))
fig, axes = plt.subplots(nRows, nCols, figsize=(4 * nCols, 3.2 * nRows))
axes = np.atleast_1d(axes).ravel()
for ax in axes:
    ax.axis("off")
for ax, idx in zip(axes, pick):
    drawImage(ax, preds[idx], confMain)
fig.suptitle(f"Hardest {len(pick)} images (blue = prediction, orange dashed = ground truth)", y=1.0)
fig.tight_layout()
fig.savefig(outputPath / "gallery.png")
plt.show()

## 8. Labeled evaluation (runs only if `labelsDir` is set)

Matching rule: predictions are sorted by confidence. Each prediction matches the unmatched ground truth box with the highest IoU, if IoU >= `iouMatch`. Unmatched predictions are false positives. Unmatched ground truth boxes are misses. The evaluation is class-agnostic, which is correct for a single-class Shahed-136 model. Images with no label file or an empty label file count as negatives.

In [ ]:
def iouMatrix(a, b):
    # a: (n,4), b: (m,4) pixel xyxy
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    areaA = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    areaB = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (areaA[:, None] + areaB[None, :] - inter + 1e-9)


if labelsDir is None:
    print("labelsDir is None: skipping labeled evaluation.")
else:
    assert Path(labelsDir).exists(), f"Labels folder not found: {labelsDir}"
    matchRows = []   # one row per prediction: image, conf, isTp
    gtPerImage = {}
    missBest = {}    # best confidence of any prediction on each GT box (for miss analysis)

    for p in preds:
        gt = loadGt(p["name"], p["width"], p["height"])
        gtPerImage[p["name"]] = len(gt)
        order = np.argsort(-p["conf"])
        boxes = p["xyxy"][order]
        confs = p["conf"][order]
        ious = iouMatrix(boxes, gt)
        used = np.zeros(len(gt), dtype=bool)
        for i in range(len(boxes)):
            isTp = False
            if len(gt):
                cand = np.where(~used)[0]
                if len(cand):
                    j = cand[np.argmax(ious[i, cand])]
                    if ious[i, j] >= iouMatch:
                        used[j] = True
                        isTp = True
            matchRows.append({"image": p["name"], "conf": float(confs[i]), "isTp": isTp})
        missBest[p["name"]] = int((~used).sum())

    dfMatch = pd.DataFrame(matchRows)
    nGt = sum(gtPerImage.values())
    print(f"Ground truth boxes: {nGt} in {sum(1 for v in gtPerImage.values() if v > 0)} images")

    evalRows = []
    for t in sorted(set(confSweep + [confMain])):
        sel = dfMatch[dfMatch["conf"] >= t]
        tp = int(sel["isTp"].sum())
        fp = int((~sel["isTp"]).sum())
        fn = nGt - tp
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / nGt if nGt else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        evalRows.append({"conf": t, "TP": tp, "FP": fp, "FN": fn,
                         "precision": prec, "recall": rec, "F1": f1})
    dfEval = pd.DataFrame(evalRows)
    display(dfEval.round(3))
    dfEval.to_csv(outputPath / "labeledEval.csv", index=False)

    # AP at IoU = iouMatch (all-point interpolation)
    ranked = dfMatch.sort_values("conf", ascending=False).reset_index(drop=True)
    cumTp = ranked["isTp"].cumsum().values
    cumFp = (~ranked["isTp"]).cumsum().values
    precCurve = cumTp / np.maximum(cumTp + cumFp, 1)
    recCurve = cumTp / max(nGt, 1)
    mrec = np.concatenate([[0.0], recCurve, [1.0]])
    mpre = np.concatenate([[1.0], precCurve, [0.0]])
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
    print(f"AP@{iouMatch} (class-agnostic): {ap:.3f}")

    # Best F1 threshold
    bestRow = dfEval.loc[dfEval["F1"].idxmax()]
    print(f"Best F1 among tested thresholds: {bestRow['F1']:.3f} at conf = {bestRow['conf']}")

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.plot(recCurve, precCurve, color=attCobalt, linewidth=2)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Precision-recall (AP@{iouMatch} = {ap:.3f})")
    fig.tight_layout()
    fig.savefig(outputPath / "precisionRecall.png")
    plt.show()

    # Error analysis at the main threshold
    fpImages = (dfMatch[(dfMatch["conf"] >= confMain) & (~dfMatch["isTp"])]
                .groupby("image").size().sort_values(ascending=False))
    print("\nImages with false positives at main threshold:")
    display(fpImages.head(15).to_frame("falsePositives"))

    tpAtMain = dfMatch[(dfMatch["conf"] >= confMain) & (dfMatch["isTp"])].groupby("image").size()
    missed = [n for n, g in gtPerImage.items() if g > 0 and tpAtMain.get(n, 0) < g]
    print(f"Images with at least 1 missed ground truth box at main threshold: {len(missed)}")
    print(missed[:20])

## 9. Optional: Ultralytics built-in validation

Use this if you have a dataset YAML for the real images (`path`, `val` or `test`, `names`). It reports mAP50, mAP50-95, precision, and recall with the library's own matching. Set `valYaml` to run it.

In [ ]:
valYaml = None  # example: "/users/apgoel/data/real_eval.yaml"

if valYaml is None:
    print("valYaml is None: skipping model.val().")
else:
    metrics = model.val(data=valYaml, split="test", conf=0.001, iou=0.6, imgsz=imgSize,
                        project=str(outputPath), name="ultralyticsVal", exist_ok=True)
    print(f"mAP50:     {metrics.box.map50:.3f}")
    print(f"mAP50-95:  {metrics.box.map:.3f}")
    print(f"Precision: {metrics.box.mp:.3f}")
    print(f"Recall:    {metrics.box.mr:.3f}")

## 10. Save annotated images and a console-style summary

In [ ]:
annotDir = outputPath / "annotated"
annotDir.mkdir(exist_ok=True)
for p in preds:
    fig, ax = plt.subplots(figsize=(8, 6))
    drawImage(ax, p, confMain)
    fig.savefig(annotDir / f"{Path(p['name']).stem}_pred.png", bbox_inches="tight")
    plt.close(fig)

print("--- Detection Summary ---")
for _, row in dfImages.iterrows():
    if row["nDet"] > 0:
        print(f"[detected] {row['image']}: {int(row['nDet'])} target(s), max conf {row['maxConf']:.2f}")
    else:
        print(f"[missed]   {row['image']}: no target (best score {row['maxConfAnyThreshold']:.2f})")
print(f"\nFinished: {nDetected}/{nImages} images had detections at conf {confMain}.")
print(f"Outputs saved to: {outputPath}")